In [1]:
import sys

if sys.platform == "win32":
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_community.utilities import SQLDatabase
from langchain.agents import AgentState
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage, AIMessage
from langgraph.types import Command
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from pathlib import Path

uvx_path = Path.home() / ".local" / "bin" / "uvx.exe"

print(uvx_path)
print(uvx_path.exists())

C:\Users\25787\.local\bin\uvx.exe
True


In [10]:
import asyncio
import traceback
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient


async def load_tools(name, connection):
    client = MultiServerMCPClient({name: connection})
    try:
        tools = await asyncio.wait_for(
            client.get_tools(),
            timeout=30,
        )
        print(f"[{name}] 成功，工具：{[tool.name for tool in tools]}")
        return tools
    except Exception:
        print(f"[{name}] 连接失败：")
        traceback.print_exc()
        return []


time_tools = await load_tools(
    "time",
    {
        "transport": "stdio",
        "command": str(Path.home() / ".local" / "bin" / "uvx.exe"),
        "args": [
            "mcp-server-time",
            "--local-timezone=Asia/Shanghai",
        ],
    },
)

travel_tools = await load_tools(
    "travel_server",
    {
        "transport": "streamable_http",
        "url": "https://mcp.kiwi.com",
    },
)

tools = time_tools + travel_tools

[time] 成功，工具：['get_current_time', 'convert_time']
[travel_server] 连接失败：


In [11]:
# MCP客户端，包含Time、Kiwi两个MCP
client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "http",
                "url": "https://mcp.kiwi.com"
        },
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

tools = await client.get_tools()